# Surface fitting with Blended Chart Surfaces (BCS)

This notebook fits a BCS to a target shape given as a signed distance function (SDF): either an analytic SDF or a pre-trained DeepSDF network in `sdf_weights/surfaces/`. Starting from a coarse proxy mesh, it learns per-vertex polynomial coefficients so that the blended surface lies on the zero level set of the SDF.

## How to run

1. **Environment.** Use the project's conda environment (Python 3.8 with PyTorch, Open3D, trimesh, libigl). The notebook can be run from anywhere inside the repository: the first cell switches to the repository root.
2. **Device.** The setup cell uses `device = 'mps'` (Apple Silicon). On other machines, change it to `'cuda'` or `'cpu'`.
3. **Choose a shape** in section 1 by setting `config_filepath` to one of the configs in `configs/surfaces/`. Optionally set `coeffs_path` to continue from previously trained coefficients.
4. **Run all cells in order.** Section 4 calls `scripts/precomputation.py` in a subprocess (with the same Python as this kernel) to precompute the training samples; this can take a few minutes for 500-face meshes.
5. **Saving.** `testing = True` (section 1) disables saving checkpoints and plots *during* training. The final section always saves the trained coefficients to `models/surfaces/<proxy>.pth` and renders the result.

**Outputs:**
- `results/<shape>/<settings>__<date>/`: the config, the proxy mesh and (if `testing = False`) per-epoch checkpoints and plots
- `models/surfaces/<proxy>.pth`: the final polynomial coefficients
- `rendering/rendering_results/<shape>/`: the final renders

Open3D viewer windows open during visualisation. Close each one to let the notebook continue.

## Setup

In [ ]:
# Run from the repository root, so that `import bcs` and the relative
# data/, configs/ and sdf_weights/ paths work wherever this notebook lives.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bcs').is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
from torch import autograd as Grad

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import matplotlib.tri as mtri
import matplotlib
from IPython.display import display, clear_output


import ipywidgets as widgets
from IPython.display import display
import trimesh

import random
import math

import time
import os
import json

import copy
import subprocess
import shutil

import open3d as o3d
print(o3d.__path__)
print('Using open3d version',o3d.__version__)
# Suppress Open3D warnings
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)


from bcs.visuals import *

from bcs.utils import *
from bcs.mesh_processing import *

from bcs import differential
import importlib
importlib.reload(differential)
from bcs.differential import *

two_pi = 2*torch.pi
eps = 1.0e-6

diffmod = DifferentialModule()

# Set the device that we will create the BCS on

device = 'mps'

print(f"Using device: {device}")

coeffs_path = None

## 1. Choose a shape
Loads the config and, for DeepSDF targets, the SDF network. The next cell checks that the coarse proxy mesh lies close to the zero level set of the SDF.

In [ ]:
# Choose the surface to fit. Each config in configs/surfaces/ specifies the target
# SDF, the coarse proxy mesh, the blend function, the polynomial degree and the
# training settings.
config_filepath = "configs/surfaces/double_torus500-inv-exp.json"

# Other configs you can swap in:
#   inv-exp blending:  arm500-inv-exp, bob500-inv-exp, dice500-inv-exp, dice520-inv-exp,
#                      double_torusMC436-inv-exp, fandisk500-inv-exp, fertility500-inv-exp,
#                      gear500-inv-exp, igea500-cubic, oloid100-inv-exp, rippley400,
#                      twisted_torus200, twisted_torus500, wobbly_torus500-diffmesh
#   trig blending:     bob400, bob500, bob500-trig, fertility300_no_posenc,
#                      lion-reference200, mobius220, sphere100, urchin210
#   experimental:      fertility500-djuren-mix (Djuren et al.'s blending and one-ring coords
#                      mixed into BCS, NOT their full method; see notebooks/djuren/)

# Optional: start from previously trained polynomial coefficients instead of
# training from scratch, e.g. coeffs_path = "models/surfaces/double_torus500.pth"
coeffs_path = None

visualise = True        # show the BCS before training (section 3)


with open(config_filepath, "r") as f:
    config_dict = json.load(f)
    namestring = stringify_config(config_dict)
    print('Namestring will be ', namestring)
    
    shape_name = config_dict['shape-name']
    surface_config = config_dict['surface-config']
    training_config = config_dict['training-config']



# Which DeepSDF architecture the checkpoint was trained with. If the
# config specifies "model_variant" directly (see
# implicit_reps.DEEPSDF_MODEL_VARIANTS for the options), that's used;
# otherwise it's derived from the older "posenc"/"activation" fields for
# backward compatibility with existing config files.
model_variant = training_config.get('model_variant', None)
posenc = training_config.get('posenc', True)
activation = training_config.get('activation', None)


if 'sdf_weights_path' in training_config.keys():
    DEEPSDF_MODEL = load_deepsdf_model(
        ckpt_path='sdf_weights/surfaces/' + training_config['sdf_weights_path'],
        device=device,
        model_variant=model_variant,
        posenc=posenc,
        activation=activation,
    )
else:
    DEEPSDF_MODEL = None

if 'sdf_transition_width' in training_config.keys():
    transition_width = training_config['sdf_transition_width']
else:
    transition_width=0.0

testing = True          # set False to enable saving
save    = not testing


In [ ]:
# --- Sanity check: evaluate the implicit function on the proxy mesh's
# vertices. The proxy is meant to approximate the zero level set of the
# SDF, so these values should be small.
proxy_filepath = 'data/surfaces/' + surface_config['coarse_patches_id'] + '.obj'
proxy_mesh = trimesh.load(proxy_filepath, process=False)

proxy_verts = torch.tensor(proxy_mesh.vertices, dtype=torch.float32, device=device).unsqueeze(0)

with torch.no_grad():
    proxy_sdf_vals = sdf(
        proxy_verts,
        training_config['sdf_id'],
        squared=False,
        model=DEEPSDF_MODEL,
        transition_width=transition_width
    )

print(f"Implicit function value on {proxy_mesh.vertices.shape[0]} proxy vertices ({proxy_filepath}):")
print(f"  mean |sdf| = {proxy_sdf_vals.abs().mean().item():.6f}")
print(f"  max  |sdf| = {proxy_sdf_vals.abs().max().item():.6f}")

## 2. Build the BCS
Creates the BCS from the coarse proxy mesh and loads `coeffs_path` if one was given.

In [ ]:
from bcs import surface
importlib.reload(surface)
from bcs.surface import BCS_fast

# instantiate
bcs = BCS_fast( surface_config, device=device)

if not coeffs_path is None:
    bcs.load_poly_coeffs(coeffs_path)


## 3. Visualise the initial surface, before optimisation (optional)
Skipped when `visualise = False`.

In [ ]:
################# visualise BCS ##########################


if visualise==True:
    
    from bcs import visuals
    importlib.reload(visuals)
    from bcs.visuals import *
    
    from bcs import differential
    
    importlib.reload(differential)
    from bcs.differential import *
    
    from bcs import mesh_processing
    importlib.reload(mesh_processing)
    from bcs.mesh_processing import *
    
    
    vis_start_time = time.time()
    
    bcs_visualiser = BCS_visualiser(
        bcs, mesh_res=4, shape_name=shape_name,
        sdf_id=training_config['sdf_id'], sdf_model=DEEPSDF_MODEL, transition_width=transition_width,
        error_scale_const = 500
    )
    
    # What to visualise. Options: 'one-colour', 'unblended', 'normals', 'error',
    # 'param', 'blend', 'bary', 'angle', 'curv', 'default'
    settings = ['one-colour', 'unblended', 'normals', 'error']

    bcs_visualiser.compute_quantities(settings=settings, degree = 2)
    
    if device == 'mps':
        torch.mps.synchronize()
    vis_end_time = time.time()
    
    print('Vis took', vis_end_time - vis_start_time, 'seconds.')


In [ ]:
if visualise==True:
    bcs_visualiser.show_bcs(settings=settings, show_on_coarse=False, title_prefix='BEFORE optimisation')
    


## 4. Precompute training samples
Runs `scripts/precomputation.py` for the chosen config and loads the result.

In [ ]:

samples_filename=None ### Do not load samples from a file because the precomputation will be different whenever settings change.
if samples_filename == None:

    subprocess.run(
    [
        sys.executable, "scripts/precomputation.py",
        "--output-filepath", "data/precomputation_results/temp-samples.pth",
        "--config-filepath", config_filepath
    ],
    check=True
)
    precomputed_training_data = torch.load('data/precomputation_results/temp-samples.pth')

else:
    precomputed_training_data = torch.load(samples_filename)

print((precomputed_training_data['poly_basis_gradient']**2).sum())
print((precomputed_training_data['blend_weights_gradient']**2).sum())
print((precomputed_training_data['onering_coords_gradient']**2).sum())

print(precomputed_training_data['all_Jc_pinv'].shape)
print(precomputed_training_data['all_J_proxy'].shape)


## 5. Training setup
Initial training state, the training loop definition, the optimiser and learning-rate schedule (all settings come from the config), and the results folder.

In [ ]:
############### initial training settings #####################

epoch=0

losses=[]
pos_losses=[]
normals_regs=[]
distortion_regs=[]
fractional_epochs=[]
lrs=[]

epochs=[]
times=[]
num_patches = bcs.num_facepatches

loss=99999

In [ ]:
def save_epoch_artifacts(
    out_dir,
    epoch,
    fig,
    info_str
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- save plot ----
    plot_path = out_dir / f"epoch_{epoch:05d}.png"
    if fig is not None:
        fig.savefig(plot_path, dpi=150)
        plt.close(fig)

    # ---- save text ----
    txt_path = out_dir / f"epoch_{epoch:05d}.txt"
    with open(txt_path, "w") as f:
        f.write(info_str)




def train_batch_loop(
    lr=0.01,
    batch_size=10,
    min_loss=0.0005,
    normals_reg_coeff=0.0,
    distortion_reg_coeff=0.0,
    batch_print_rate=100,
    batch_plot_rate=10,
    epoch_print_rate=1,
    epoch_save_rate=1,
    epoch_plot_rate=1,
    plot=False,
    flag='poly',
    just_onering=False,
    degree=3,
    max_epochs=1000,
    verify_gradient=False,
    area_weighting=False,
    save=True
):
    global optimizer, scheduler, losses, pos_losses, normals_regs, distortion_regs
    global times, fractional_epochs, epoch, loss, num_patches, eps, sdf_id
    global lrs  # learning-rate history

    # Ensure lr list exists
    try:
        lrs
    except NameError:
        lrs = []

    if degree is None:
        degree = bcs.degree

    if degree > bcs.degree:
        raise ValueError(
            f"Sorry, this BCS supports only degree {bcs.degree} (requested degree {degree})."
        )

    if distortion_reg_coeff > 0.0 or normals_reg_coeff > 0.0 or area_weighting==True:
        compute_jacobian=True
    else:
        compute_jacobian=False

    # Patch selection
    if just_onering:
        select_patch_indices = torch.tensor(
            bcs.onerings[0]['triangles'], dtype=torch.long, device=bcs.device
        )
    else:
        select_patch_indices = torch.arange(
            bcs.F.shape[0], dtype=torch.long, device=bcs.device
        )

    seqs = bcs.F.shape[0]
    total_num_points = precomputed_training_data['equilateral_triangle_samples'].shape[1]


    start_time = time.time()

    # ---- begin epochs ----
    while loss > min_loss and epoch < max_epochs:

        perm = torch.randperm(total_num_points)
        epoch_loss = 0.0

        last_fig = None
        last_info_str = ""

        # ---- batches ----
        for i in range(0, total_num_points, batch_size):

            # ======================================================
            # TRUE GET-DATA TIME
            # ======================================================
            get_t0 = time.time()

            idx = perm[i:i + batch_size]
            optimizer.zero_grad()

            precomputed_batch_data = bcs.get_batch(precomputed_training_data, idx)
            batch_t = precomputed_batch_data['equilateral_triangle_samples']

            if compute_jacobian:
                for k in ['blend_weights', 'onering_coords', 'poly_basis']:
                    precomputed_batch_data[k] = (
                        precomputed_batch_data[k]
                        .detach()
                        .clone()
                        .requires_grad_(True)
                    )
                

            torch.mps.synchronize()
            get_data_time = time.time() - get_t0


            # ======================================================
            # TRUE FORWARD TIME
            # ======================================================
            torch.mps.synchronize()
            fwd_t0 = time.time()

            output_train = bcs(precomputed_batch_data, degree=degree)

            torch.mps.synchronize()
            fwd_time = time.time() - fwd_t0


            # ======================================================
            # LOSS COMPUTATION
            # ======================================================
            selected_pts = output_train[select_patch_indices, :, :].to(device)
            cur_sdf = sdf(
                selected_pts.reshape(1, -1, 3),
                training_config['sdf_id'],
                squared=False,
                model=DEEPSDF_MODEL,
                transition_width=transition_width
            )
            cur_udf = cur_sdf.abs()
            




            # Normals / distortion regularization
            if compute_jacobian==True:   ## 


                J_proxy = precomputed_batch_data['all_J_proxy']
                Jc_pinv = precomputed_batch_data['all_Jc_pinv']


                #print('gradient via blend')
                grad_via_blend = diffmod.gradient(out = output_train, wrt = precomputed_batch_data['blend_weights']) @ precomputed_batch_data['blend_weights_gradient']
                #print('grad blend', grad_via_blend.shape)

                

                d_out_d_poly = diffmod.gradient(out = output_train, wrt = precomputed_batch_data['poly_basis'])
                d_poly_d_oc = precomputed_batch_data['poly_basis_gradient']
                d_oc_dx = precomputed_batch_data['onering_coords_gradient']



                grad_via_poly = torch.einsum('fbRsA, fbsRx, fsAxr -> fsRr', d_out_d_poly, d_poly_d_oc, d_oc_dx)
                jacobian = grad_via_blend + grad_via_poly
                
                ####################### TEST ONLY!!! #####################

                if verify_gradient:
                    cur_x = precomputed_batch_data['equilateral_triangle_samples'].detach()
                    cur_x.requires_grad=True
                    batch_data_with_tracked_gradients = bcs.precompute_data_from_samples( cur_x, detached=False)
    
                    output_train_with_tracked_gradients = bcs(batch_data_with_tracked_gradients, degree=degree)
    
                    autograd_jacobian = diffmod.gradient(out=output_train_with_tracked_gradients, wrt = cur_x)
    
                    for jacobian in [ jacobian]:
                        print( 'difference between automatic and manual jacobians (sum of squared differences)', ((jacobian-autograd_jacobian)**2).sum() )
                        print('tensor sizes', jacobian.shape, autograd_jacobian.shape)
                        print('sum of squared values of each, for scale reference:')
                        print(((jacobian)**2).sum())
                        print(((autograd_jacobian)**2).sum())

                        diff = jacobian - autograd_jacobian

                        print("max abs:", diff.abs().max())
                        print("mean abs:", diff.abs().mean())
                        
                        print("relative max:",
                              (diff.abs() / (autograd_jacobian.abs() + 1e-12)).max())
                    
                    
                
                ######################################

                jacobian = jacobian
                
                output_normals, jacobian = diffmod.compute_normals(
                    out=output_train,
                    wrt=batch_t,
                    jacobian = jacobian,
                    return_grad=True
                )

                


                #print('output_normals', output_normals)

            if normals_reg_coeff > 0.0:


                grad_sdf = torch.autograd.grad(
                    outputs=cur_sdf,
                    inputs=output_train,
                    grad_outputs=torch.ones_like(cur_sdf),
                    create_graph=False, #important that this is false.
                    retain_graph=True
                )[0]


                grad_sdf = grad_sdf         / (grad_sdf.norm(dim=-1, keepdim=True) + 1e-4)
                
                normals_alignment = (output_normals * grad_sdf).sum(-1)    ##### .clamp(min=-1, max=1

                local_normals_loss = (-normals_alignment + 1.0)
                normals_loss = local_normals_loss.mean()
                normals_reg = normals_reg_coeff * normals_loss
            else:
                normals_reg = torch.tensor(0.0, device=device)



            if distortion_reg_coeff > 0.0 or area_weighting==True:
                #I_E, I_F, I_G = diffmod.compute_FFF(
                #    out=output_train,
                #    wrt=batch_t,
                #    jacobian=jacobian,
                #    normals=output_normals )

                #print('jac, J_proxy, Jc_pinv', jacobian.shape, J_proxy.shape, Jc_pinv.shape)

                full_jacobian = jacobian @ J_proxy.unsqueeze(1) @ Jc_pinv.unsqueeze(1)    # chain rule

                
                
                M3 = full_jacobian.transpose(-1, -2) @ full_jacobian
                M2 = Jc_pinv.unsqueeze(1) @ M3 @ Jc_pinv.transpose(-1, -2).unsqueeze(1)  # (2×2)
                I_E, I_G, I_F = M2[:,:,0,0], M2[:,:,0,1], M2[:,:,1,1]
                
                trace = I_E + I_G
                det = I_E * I_G - I_F * I_F

            if distortion_reg_coeff > 0.0:    
                distortion_reg = distortion_reg_coeff * (trace + trace / (det + eps)).mean()
            else:
                distortion_reg = torch.tensor(0.0, device=device)

            if area_weighting==True:
                #print(cur_udf.shape, det.shape)

                w = det.reshape(-1) / ( det.mean() + eps)
                position_loss = (cur_udf*w).mean()
            else:
                position_loss = cur_udf.mean()
            

            batch_loss = position_loss + normals_reg + distortion_reg

            # ---- accumulate real epoch loss ----
            epoch_loss += batch_loss.item() * batch_t.shape[1]

            # ======================================================
            # SCHEDULER STEP + RECORD LR
            # ======================================================
            
            current_lr = optimizer.param_groups[0]['lr']
            


            # ======================================================
            # TRUE BACKWARD TIME
            # ======================================================
            torch.mps.synchronize()
            bwd_t0 = time.time()

            batch_loss.backward()

            torch.mps.synchronize()
            bwd_time = time.time() - bwd_t0


            # ======================================================
            # TRUE OPT TIME
            # ======================================================
            torch.mps.synchronize()
            opt_t0 = time.time()

            optimizer.step()
            scheduler.step()

            torch.mps.synchronize()
            opt_time = time.time() - opt_t0


            # total batch time
            batch_time = get_data_time + fwd_time + bwd_time + opt_time


            # ---- logs ----
            times.append(batch_time)
            losses.append(batch_loss.item())
            pos_losses.append(position_loss.item())
            normals_regs.append(normals_reg.item())
            distortion_regs.append(distortion_reg.item())
            fractional_epochs.append(epoch + i / total_num_points)
            lrs.append(current_lr)


            # ======================================================
            # OPTIONAL PLOTTING (NOW 3 PANELS)
            # ======================================================
            if (i // batch_size) % batch_plot_rate == 0 and plot:
                try:
                    plt.close(fig)
                except:
                    pass

                clear_output(wait=True)

                fig, axs = plt.subplots(3, 1, figsize=(10, 10))

                # ---- Panel 1: Loss ----
                axs[0].plot(fractional_epochs, np.log10(losses), label='loss')
                axs[0].plot(fractional_epochs, np.log10(pos_losses), label='pos loss')

                if normals_reg_coeff > 0:
                    axs[0].plot(fractional_epochs, np.log10(normals_regs), label='normals reg')
                if distortion_reg_coeff > 0:
                    axs[0].plot(fractional_epochs, np.log10(distortion_regs), label='distortion reg')

                axs[0].legend()
                axs[0].set_xlabel('Epoch')
                axs[0].set_ylabel('Log10 Loss')
                axs[0].set_title('Log10 Losses')

                axs[0].text(0.8, 0.9, f"Batch Loss: {batch_loss.item():.8f}",
                            transform=axs[0].transAxes,
                            ha='right', va='bottom')

                # ---- Panel 2: Timing ----
                axs[1].plot(fractional_epochs, times)
                axs[1].set_xlabel('Epoch')
                axs[1].set_title('True Batch Times')

                axs[1].text(0.98, 0.90, f"Batch Time: {batch_time:.4f}s",
                            transform=axs[1].transAxes, ha='right')
                axs[1].text(0.98, 0.75, f"Fwd Time: {fwd_time:.4f}s",
                            transform=axs[1].transAxes, ha='right')
                axs[1].text(0.98, 0.60, f"Bwd Time: {bwd_time:.4f}s",
                            transform=axs[1].transAxes, ha='right')
                axs[1].text(0.98, 0.45, f"Get Data: {get_data_time:.4f}s",
                            transform=axs[1].transAxes, ha='right')
                axs[1].text(0.98, 0.30, f"Opt Time: {opt_time:.4f}s",
                            transform=axs[1].transAxes, ha='right')

                # ---- Panel 3: Learning Rate ----
                axs[2].plot(fractional_epochs, lrs)
                axs[2].set_xlabel('Epoch')
                axs[2].set_title('Learning Rate')
                axs[2].text(0.98, 0.85, f"LR: {current_lr:.6e}",
                            transform=axs[2].transAxes, ha='right')

                plt.tight_layout()
                display(fig)

                

                info_str = (
                    f"Epoch {epoch}\n"
                    f"Batch {i}\n"
                    f"Loss: {batch_loss:.8f}\n"
                    f"  pos:       {position_loss:.8f}\n"
                    f"  normals:   {normals_reg:.8f}\n"
                    f"  distort:   {distortion_reg:.8f}\n"
                    f"mean UDF: {cur_udf.mean():.8f}\n"
                    f"learning rate: {current_lr:.8f}\n"
                    f"\nTiming:\n"
                    f"  batch time {batch_time:.4f}s\n"
                    f"  fwd {fwd_time:.4f}s | bwd {bwd_time:.4f}s | "
                    f"data {get_data_time:.4f}s | opt {opt_time:.4f}s\n" )

                last_fig = fig
                last_info_str = info_str
                
                print(info_str)
    


        # ---- End of epoch ----
        loss = torch.tensor( epoch_loss / total_num_points )

        if epoch % epoch_print_rate == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.8f}")

        if epoch % epoch_save_rate == 0 and save:
            ckpt_dir = results_dir 
        
            ckpt_path = ckpt_dir / f"epoch_{epoch:05d}.pth"
            bcs.save_poly_coeffs(str(ckpt_path))
        
            save_epoch_artifacts( out_dir=ckpt_dir,epoch=epoch,fig=last_fig,info_str=last_info_str)
        
            print(f"Saved checkpoint + plot + log for epoch {epoch}.")

        epoch += 1
    end_time=time.time()
    
    print(f"Took {int((end_time-start_time)/60)}m {(end_time-start_time)%60:.2f}s")


In [ ]:


#Training settings are specified in the config file.
initial_lr = training_config['initial_lr']
MIN_LR = training_config['min_lr']
max_epochs = training_config['max_epochs']
per_face_batch_size = training_config['per_face_batch_size']

area_weighting = training_config['area_weighting']
print(area_weighting)
distortion_reg_coeff = training_config['distortion_reg_coeff']
normals_reg_coeff = training_config['normals_reg_coeff']

num_samples_per_face = training_config['num_samples_per_face']


optimizer = torch.optim.AdamW(bcs.parameters(), lr=initial_lr)


T_max = (num_samples_per_face/per_face_batch_size)*max_epochs

# Initialize CosineAnnealingLR
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=T_max, 
    eta_min=MIN_LR
)

In [ ]:
from datetime import datetime
from pathlib import Path

import shutil


# Build readable results name
results_namestring = (
    stringify_config(config_dict) 
    + "__"
    + datetime.now().strftime("%Y-%m-%d_%H-%M")
    
)


# Build directory path
results_dir = Path("results") / shape_name / results_namestring

# Create directories (no error if they already exist)
results_dir.mkdir(parents=True, exist_ok=True)



json.dump(config_dict, open(results_dir / "config.json", "w"), indent=2)




shutil.copy2('data/surfaces/'+surface_config['coarse_patches_id']+'.obj', results_dir / 'proxy.obj')





## 6. Train

In [ ]:
train_batch_loop(lr=None,  degree=None, batch_size=per_face_batch_size , min_loss=5e-5, max_epochs=max_epochs, normals_reg_coeff = normals_reg_coeff, distortion_reg_coeff = distortion_reg_coeff, batch_plot_rate=10, plot=True, verify_gradient=False, area_weighting=area_weighting, save=save)

## 7. Save and visualise the result

In [ ]:
save = True   # save the trained coefficients and final renders

checkpoint_dir = Path('models/surfaces')
checkpoint_filename = surface_config['coarse_patches_id'] + '.pth'   # e.g. "arm500.pth"

if save:
    bcs.save_poly_coeffs(checkpoint_dir / checkpoint_filename)
    print(f"Saved polynomial coefficients to {checkpoint_dir / checkpoint_filename}")

################ visualise bcs ##################

from bcs import visuals
import importlib
importlib.reload(visuals)
from bcs.visuals import *

bcs_visualiser = BCS_visualiser(
    bcs, mesh_res=4, just_onering=False, show_on_coarse=False, shape_name=shape_name,
    sdf_id=training_config['sdf_id'], sdf_model=DEEPSDF_MODEL, transition_width=transition_width,
)

settings = ['one-colour', 'unblended']

bcs_visualiser.compute_quantities(settings=settings)
bcs_visualiser.show_bcs(settings=settings, save=save, title_prefix='AFTER optimisation')   # output_dir omitted -> rendering/rendering_results/<shape_name>/
